# EXP-09: BPTI–Trypsin Association/Dissociation Kinetics
**PMF-based TST/Kramers’ rate constant estimation from EXP-04 US data**

- **Expected:** k_on ≈ 10⁶ M⁻¹s⁻¹, k_off ≈ 6×10⁻⁸ s⁻¹
- **PASS:** both within 2 orders | **MARGINAL:** one within 3 | **FAIL:** >3 orders
- **Depends on:** EXP-04 umbrella timeseries + PMF data on Drive

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np
from scipy.signal import argrelextrema

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-09'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['analysis','figures']: (WORK_DIR/d).mkdir(parents=True, exist_ok=True)

EXP04 = Path('/content/drive/MyDrive/v3_gpu_results/EXP-04')
assert EXP04.exists(), 'Run EXP-04 first'
logging.basicConfig(level=logging.INFO)

from src.config import KCAL_TO_KJ
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('Imports OK')

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Load EXP-04 PMF data
pmf_file = list(EXP04.glob('**/pmf_data.npz'))
assert pmf_file, 'EXP-04 pmf_data.npz not found'
data = np.load(str(pmf_file[0]))
xi = data['xi']
pmf_kcal = data['pmf_kcal']
fm = np.isfinite(pmf_kcal)
print(f'PMF: {len(xi)} bins, range [{xi[0]:.2f}, {xi[-1]:.2f}] nm')

# Also load EXP-04 results for \u0394G
res04_files = list(EXP04.glob('**/results.json'))
dg_bind = -18.0
if res04_files:
    with open(str(res04_files[0])) as f:
        r04 = json.load(f)
    dg_bind = r04.get('dg_wham_kcal_mol', -18.0)
print(f'\u0394G_bind = {dg_bind:.1f} kcal/mol')

In [ ]:
# Identify bound-state minimum and transition state
idx_min = np.argmin(pmf_kcal[fm])
idx_min_global = np.where(fm)[0][idx_min]
xi_min = xi[idx_min_global]

# Find TS: first local maximum after the minimum
search = pmf_kcal[idx_min_global:]
maxima = argrelextrema(search, np.greater, order=5)[0]
if len(maxima) > 0:
    idx_ts = maxima[0] + idx_min_global
else:
    # Fallback: midpoint region
    idx_ts = idx_min_global + len(search) // 2

xi_ts = xi[idx_ts]
barrier = pmf_kcal[idx_ts] - pmf_kcal[idx_min_global]
print(f'Bound: \u03be_min={xi_min:.2f} nm, PMF={pmf_kcal[idx_min_global]:.2f} kcal/mol')
print(f'TS: \u03be\u2021={xi_ts:.2f} nm, \u0394W\u2021={barrier:.2f} kcal/mol')

In [ ]:
# Estimate D(\u03be) from umbrella timeseries near TS
us_npz = list(EXP04.glob('**/umbrella_timeseries.npz'))
D_xi = 0.05  # default nm\u00b2/ns
tau_corr = 0.01  # ns

if us_npz:
    us_data = np.load(str(us_npz[0]), allow_pickle=True)
    # Find the window closest to TS
    if 'window_centers' in us_data:
        wc = us_data['window_centers']
        ts_win = int(np.argmin(np.abs(wc - xi_ts)))
        # Look for xi timeseries for this window
        key = f'xi_window_{ts_win:03d}'
        if key in us_data:
            xi_ts_series = us_data[key]
            xi_fluct = xi_ts_series - np.mean(xi_ts_series)
            var_xi = np.var(xi_fluct)
            ac = np.correlate(xi_fluct, xi_fluct, mode='full')[len(xi_fluct)-1:]
            ac /= ac[0]
            tau_idx = np.argmax(ac < 1/np.e)
            dt_ps = 1.0  # 1 ps save interval
            tau_corr = max(tau_idx * dt_ps / 1000, 0.01)
            D_xi = var_xi / tau_corr

print(f'D(\u03be\u2021) = {D_xi:.4f} nm\u00b2/ns, \u03c4_corr = {tau_corr:.3f} ns')

In [ ]:
# Kinetic rate constants
R = 1.987e-3  # kcal/(mol*K)
T = 310.0
kT = R * T

# PMF curvature
dx = xi[1] - xi[0]
Wpp = np.gradient(np.gradient(pmf_kcal, dx), dx)
omega_min = np.sqrt(max(abs(Wpp[idx_min_global]) * KCAL_TO_KJ, 1e-3))
omega_ts = np.sqrt(max(abs(Wpp[idx_ts]) * KCAL_TO_KJ, 1e-3))

# Kramers (high friction)
gamma = kT / D_xi if D_xi > 0 else 1.0
koff_kramers = (omega_min * omega_ts) / (2 * np.pi * gamma) * np.exp(-barrier / kT)

# TST
nu = 1e12  # s^-1 attempt frequency
koff_tst = nu * np.exp(-barrier / kT)

# kon from detailed balance: Kd = koff/kon, Kd = exp(dG/kT) / 1M
Kd = np.exp(dg_bind / kT)  # M
kon_kramers = koff_kramers / Kd if Kd > 0 else 0
kon_tst = koff_tst / Kd if Kd > 0 else 0

# Experimental values
kon_exp, koff_exp = 1e6, 6e-8

print(f'koff (Kramers): {koff_kramers:.2e} s\u207b\u00b9')
print(f'koff (TST):     {koff_tst:.2e} s\u207b\u00b9')
print(f'kon (Kramers):  {kon_kramers:.2e} M\u207b\u00b9s\u207b\u00b9')
print(f'kon (TST):      {kon_tst:.2e} M\u207b\u00b9s\u207b\u00b9')
print(f'Kd:             {Kd:.2e} M')

In [ ]:
# Classification
log_kon_err = abs(np.log10(max(kon_kramers, 1e-20)) - np.log10(kon_exp))
log_koff_err = abs(np.log10(max(koff_kramers, 1e-20)) - np.log10(koff_exp))

if log_kon_err <= 2 and log_koff_err <= 2:
    verdict = 'PASS'
elif (log_kon_err <= 2 and log_koff_err <= 3) or (log_kon_err <= 3 and log_koff_err <= 2):
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'

dg_kinetic = kT * np.log(Kd)
thermo_consistency = abs(dg_kinetic - dg_bind)
print(f'log10 errors: kon={log_kon_err:.1f}, koff={log_koff_err:.1f}')
print(f'Thermo consistency: {thermo_consistency:.2f} kcal/mol')
print(f'Verdict: {verdict}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fig 1: PMF with barrier
ax = axes[0]
ax.plot(xi[fm], pmf_kcal[fm], 'b-', lw=2)
ax.plot(xi_min, pmf_kcal[idx_min_global], 'go', ms=10, label='Bound')
ax.plot(xi_ts, pmf_kcal[idx_ts], 'r^', ms=10, label='TS')
ax.annotate(f'\u0394W\u2021={barrier:.1f}', xy=(xi_ts, pmf_kcal[idx_ts]),
            xytext=(xi_ts+0.3, pmf_kcal[idx_ts]-2), arrowprops=dict(arrowstyle='->', color='r'), color='r')
ax.set_xlabel('\u03be (nm)'); ax.set_ylabel('PMF (kcal/mol)'); ax.set_title('PMF Barrier'); ax.legend()

# Fig 2: Rate constant comparison
ax = axes[1]
vals = [np.log10(kon_exp), np.log10(max(kon_kramers,1e-20)), np.log10(max(kon_tst,1e-20))]
ax.bar(['Exp', 'Kramers', 'TST'], vals, color=['gold','steelblue','coral'], edgecolor='black')
ax.set_ylabel('log10(kon / M\u207b\u00b9s\u207b\u00b9)'); ax.set_title('kon Estimates'); ax.grid(True, alpha=0.3)

# Fig 3: koff comparison
ax = axes[2]
vals2 = [np.log10(koff_exp), np.log10(max(koff_kramers,1e-20)), np.log10(max(koff_tst,1e-20))]
ax.bar(['Exp', 'Kramers', 'TST'], vals2, color=['gold','steelblue','coral'], edgecolor='black')
ax.set_ylabel('log10(koff / s\u207b\u00b9)'); ax.set_title('koff Estimates'); ax.grid(True, alpha=0.3)

fig.suptitle(f'EXP-09: BPTI\u2013Trypsin Kinetics \u2014 {verdict}', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
results = {
    'experiment_id': EXP_ID, 'kon_kramers': float(kon_kramers), 'koff_kramers': float(koff_kramers),
    'kon_tst': float(kon_tst), 'koff_tst': float(koff_tst),
    'kon_exp': float(kon_exp), 'koff_exp': float(koff_exp),
    'barrier_kcal': float(barrier), 'Kd_M': float(Kd),
    'thermo_consistency_kcal': float(thermo_consistency), 'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2, default=str)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)
# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))